In [22]:
import pandas as pd
import numpy as np
from deep_translator import GoogleTranslator

/Users/emma/projects/cognoscolab/myvenv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_row', None)

In [3]:
rassereno_path = '/Users/emma/projects/cognoscolab/rassereno'
math_ma_llm = '/Users/emma/projects/cognoscolab/math_anxiety_LLMs'

## MAS-IT dataframes merge

In [4]:
mas_it_2023 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2023/mas_it_2023_humans.csv')
mas_it_humans_2024 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2024/mas_it_2024_humans.csv')
mas_it_BA_2024 = pd.read_csv(f'{rassereno_path}/out/formatting/psychology_BA_2024/mas_it_2024.csv')
mas_it_2025 = pd.read_csv(f'{rassereno_path}/out/formatting/all_students_2025/mas_it_2025.csv')

In [5]:
# gets all the physics students as saves the df to csv
mas_it_fisica = mas_it_2025[mas_it_2025['participant_id'].str.startswith('fisica')].reset_index(drop=True)

mas_it_fisica.to_csv(f'{rassereno_path}/out/formatting/physics_2025/mas_it_fisica_2025.csv', index=False)
print('Number of physics students:', len(mas_it_fisica))

Number of physics students: 10


In [6]:
# gets all the high school students as saves the df to csv
mas_it_liceo_2025 = mas_it_2025[mas_it_2025['participant_id'].str.startswith('liceo')].reset_index(drop=True)

mas_it_liceo_2025.to_csv(f'{rassereno_path}/out/formatting/liceo_2025/mas_it_liceo_2025.csv', index=False)
print('Number of high school students:', len(mas_it_liceo_2025))

Number of high school students: 62


In [7]:
# gets all the university students of 2025 and saves it in a csv file
mas_it_psychology_2025 = mas_it_2025[mas_it_2025['participant_id'].str.startswith('interfacce')].reset_index(drop=True)
print('Number of psychology students of 2025:', len(mas_it_psychology_2025))

Number of psychology students of 2025: 23


In [8]:
# get the column names of the 2025 dataset
cols_2025 = mas_it_psychology_2025.columns

# restrict all dataframes to those columns before concatenation
dfs = [
    mas_it_humans_2024[cols_2025.intersection(mas_it_humans_2024.columns)],
    mas_it_2023[cols_2025.intersection(mas_it_2023.columns)],
    mas_it_BA_2024[cols_2025.intersection(mas_it_BA_2024.columns)],
    mas_it_psychology_2025[cols_2025]
]

# concatenate them
all_psycology_mas_it = pd.concat(dfs, ignore_index=True)
all_psycology_mas_it.to_csv(f'{rassereno_path}/out/formatting/all_psychology_students/mas_it_all_psy_students.csv', index=False)

print('Number of all the psychology students:', len(all_psycology_mas_it))

Number of all the psychology students: 316


## Associations and Valence dataframes merge

In [9]:
associations_valence_2023 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2023/associations_valence_2023_humans.csv')
associations_valence_2024 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2024/associations_valence_2024_humans.csv')
associations_valence_BA_2024 = pd.read_csv(f'{rassereno_path}/out/formatting/psychology_BA_2024/associations_valence_2024.csv')
associations_valence_2025 = pd.read_csv(f'{rassereno_path}/out/formatting/all_students_2025/associations_valence_task_2025.csv')
associations_valence_gpt_oss = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/associations_valence_physics.csv')

In [10]:
associations_valence_psychology_2025 = associations_valence_2025[associations_valence_2025['participant_id'].str.startswith('interfacce')]

associations_valence_fisica = associations_valence_2025[associations_valence_2025['participant_id'].str.startswith('fisica')].reset_index(drop=True)

associations_valence_fisica.to_csv(f'{rassereno_path}/out/formatting/physics_2025/associations_valence_fisica.csv', index=False)

associations_valence_liceo_2025 = associations_valence_2025[associations_valence_2025['participant_id'].str.startswith('liceo')]

associations_valence_liceo_2025.to_csv(f'{rassereno_path}/out/formatting/liceo_2025/associations_valence_liceo_2025.csv', index=False)

In [11]:
all_psychology_associations_valence = pd.concat([associations_valence_2024, associations_valence_2023, associations_valence_BA_2024, associations_valence_psychology_2025], ignore_index=True)

all_psychology_associations_valence.to_csv(f'{rassereno_path}/out/formatting/all_psychology_students/associations_valence_all_psy_students.csv', index=False)

## Mergign whole human and GPT datasets

### Human participants

In [12]:
def participant_id_mapping(mas_it_df = pd.DataFrame(), associations_valence_df = pd.DataFrame(), sample = str):
    mas_it_copy = mas_it_df.copy()
    associations_valence_copy = associations_valence_df.copy()

    for i in range(len(mas_it_copy)):
        mas_it_copy.loc[i, 'participant_id'] = f'{sample}_{i+1:03}'

    map_participant_id = {mas_it_df.loc[i, 'participant_id'] : mas_it_copy.loc[i, 'participant_id'] for i in range(len(mas_it_copy))}

    associations_valence_copy['participant_id'] = associations_valence_df['participant_id'].map(map_participant_id)

    return mas_it_copy, associations_valence_copy

In [13]:
psy_mas_it, psy_asociations_valence = participant_id_mapping(all_psycology_mas_it, all_psychology_associations_valence, 'psychology')

high_school_mas_it, high_school_associations_valence = participant_id_mapping(mas_it_liceo_2025, associations_valence_liceo_2025, 'high_school')

physics_mas_it, physics_associations_valence = participant_id_mapping(mas_it_fisica, associations_valence_fisica, 'physics')

In [14]:
all_humans_associations_valence = pd.concat([psy_asociations_valence, physics_associations_valence, high_school_associations_valence], axis=0).reset_index(drop=True)
all_humans_associations_valence.to_csv(f'{rassereno_path}/out/formatting/database/all_humans_associations_valence.csv', index=False)
all_humans_mas_it = pd.concat([psy_mas_it, physics_mas_it, high_school_mas_it], axis = 0).reset_index(drop=True)
all_humans_mas_it.to_csv(f'{rassereno_path}/out/formatting/database/all_humans_mas_it.csv', index=False)

### GPT-oss participants

In [15]:
associations_gpt_psychology = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/associations_valence_psychology.csv')
mas_it_gpt_psychology = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/mas_it_psychology.csv')

associations_gpt_physics = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/associations_valence_physics.csv')
mas_it_gpt_physics = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/mas_it_physics.csv')

associations_gpt_liceo = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/associations_valence_liceo.csv')
mas_it_gpt_liceo = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/mas_it_liceo.csv')

gpt_psychology_mas_it, gpt_psychology_associations_valence = participant_id_mapping(mas_it_gpt_psychology, associations_gpt_psychology, 'gpt_oss_psychology')

gpt_physics_mas_it, gpt_physics_associations_valence = participant_id_mapping(mas_it_gpt_physics, associations_gpt_physics, 'gpt_oss_physics')

gpt_high_school_mas_it, gpt_high_school_associations = participant_id_mapping(mas_it_gpt_liceo, associations_gpt_liceo, 'gpt_oss_high_school')

In [16]:
all_gpt_associations_valence = pd.concat([gpt_psychology_associations_valence, gpt_physics_associations_valence, gpt_high_school_associations], axis=0).reset_index(drop=True)
all_gpt_associations_valence.to_csv(f'{rassereno_path}/out/formatting/database/all_gpt_associations_valence.csv', index=False)
all_gpt_mas_it = pd.concat([gpt_psychology_mas_it, gpt_physics_mas_it, gpt_high_school_mas_it], axis = 0).reset_index(drop=True)
all_gpt_mas_it.to_csv(f'{rassereno_path}/out/formatting/database/all_gpt_mas_it.csv', index=False)

## Cue words

### Common cue words

In [17]:
common_words_university = list(set(associations_valence_2024['cue_word']) & set(associations_valence_2023['cue_word']) & set(associations_valence_BA_2024['cue_word']) & set(all_psychology_associations_valence
['cue_word']))

print(f'Common word of all university subgroups: {common_words_university}')

Common word of all university subgroups: ['statistica', 'biologia', 'esame', 'curiosità', 'ansia', 'sfida', 'matematica', 'fisica', 'insegnante', 'equazione']


In [18]:
set(associations_valence_gpt_oss['cue_word']) - set(associations_valence_2024['cue_word'])

{'creativita', 'curiosita', 'personalita', 'universita'}

In [19]:
gpt_oss_associations_valence = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/associations_valence_physics.csv')
list(set(associations_valence_liceo_2025['cue_word']) & set(all_psychology_associations_valence['cue_word']) & set(gpt_oss_associations_valence['cue_word']))

['innovazione',
 'statistica',
 'esame',
 'matematica',
 'ansia',
 'sfida',
 'fisica',
 'scienza',
 'informatica',
 'equazione',
 'scienziato',
 'benessere']

### Cue words Latex table

In [60]:
STEM_Disciplines = ['algoritmo', 'analisi', 'biologia', 'calcolo', 'dimostrazione', 'equazione','fisica', 'informatica', 'ingegneria','matematica', 'matematico/a', 'numeri', 'performance_in_matematica', 'statistica', 'tecnologia', 'test_di_matematica', 'utilità_della_matematica', 'dati', 'grafico', 'ipotesi', 'laboratorio', 'logica', 'ricerca', 'scienza', 'scienziata', 'scienziato','scoperta', 'esperimento', 'innovazione', 'neuroscienze']

Academic_Environment = ['classe', 'compagni', 'docente', 'genitore', 'genitori', 'insegnante', 'studente', 'studente/ssa', 'professore', 'lavagna', 'mappa_concettuale', 'note', 'scuola', 'università']

Evaluation = ['esame', 'esercitazione', 'prova', 'test', 'media', 'valutazione', 'voto']

Learning_and_Motivation = ['cambio_di_piani', 'compito', 'concentrazione', 'confusione', 'conoscenza', 'creatività', 'curiosità','distrazione', 'divertimento', 'errori', 'fissare_un_obiettivo', 'impegno', 'importanza', 'intelligenza', 'memorizzare', 'motivazione','pianificare', 'problema', 'programma', 'programmare', 'programmazione', 'revisionare', 'rimandare', 'scadenza', 'sfida', 'sforzo',     'soluzione', 'studiare', 'successo', 'tenere_traccia', 'futuro', 'modello', 'sapere', 'regole', 'relazione', 'sviluppo', 'passione']

Mental_Health = ['ansia', 'arte', 'attaccamento', 'benessere', 'cognizione', 'counselling', 'emozione', 'fallimento', 'frustrazione', 'imbarazzo', 'meditazione', 'mente', 'noia', 'panico', 'paura', 'personalità', 'psicologia', 'psicopatologia', 'psicoterapia', 'terapeuta', 'terapia', 'stress']

category_lists = {
    'STEM Disciplines': STEM_Disciplines,
    'Academic Environment': Academic_Environment,
    'Evaluation': Evaluation,
    'Learning and Motivation': Learning_and_Motivation,
    'Mental Health': Mental_Health
}

In [61]:
def cue_words_summary(df):
    n_cue_words = len(set(df['cue_word']))
    n_participants = len(set(df['participant_id']))

    # words = sorted(df['cue_word'].unique())
    # categories = ['STEM Disciplines', 'Academic Environment', 'Evaluation', 'Learning and Motivation', 'Mental Health']
    grouped_words = {cat: [w for w in lst if w in df['cue_word'].unique()] for cat, lst in category_lists.items()}
    
    translated_grouped_words = {}
    for cat, words_list in grouped_words.items():
        translated_grouped_words[cat] = [
            GoogleTranslator(source='it', target='en').translate(word.capitalize())
            for word in words_list
        ]

    # build a dataframe with one row and lists in the cells
    df = pd.DataFrame([translated_grouped_words])

    # df = pd.DataFrame(dict([(k, pd.Series(v)) for k,v in translated_grouped_words.items()]))
    print(f"len df {len(df)}; {n_cue_words} words for {n_participants} participants")
    # df.insert(0, 'N', np.repeat('', len(df)))
    
    return df

df_2023 = cue_words_summary(associations_valence_2023)
df_2024 = cue_words_summary(associations_valence_2024)
df_BA_2024 = cue_words_summary(associations_valence_BA_2024)
df_2025 = cue_words_summary(associations_valence_psychology_2025)

len df 1; 51 words for 57 participants
len df 1; 40 words for 70 participants
len df 1; 41 words for 166 participants
len df 1; 42 words for 23 participants


In [62]:
STEM_Disciplines = [w.capitalize() for w in ['science', 'physics', 'mathematics', 'chemistry', 'biology', 'statistics', 'electricity', 'computer', 'evolution', 'dna','experiment', 'cell', 'derivative', 'space', 'graph', 'theorem', 'trigonometry', 'integral', 'model', 'complex', 'chaos', 'solid', 'number', 'hub']]
Academic_Environment = [w.capitalize() for w in ['university', 'school', 'conference', 'lab', 'studies', 'teaching', 'student', 'degree', 'mathematician', 'prisoner']]
Evaluation = [w.capitalize() for w in ['analysis', 'game', 'word', 'loop', 'market']]
Learning_and_Motivation = [w.capitalize() for w in ['language', 'person']]
Mental_Health = [w.capitalize() for w in ['life', 'brain', 'water', 'scientific', 'cloud']]

category_lists = {
    'STEM Disciplines': STEM_Disciplines,
    'Academic Environment': Academic_Environment,
    'Evaluation': Evaluation,
    'Learning and Motivation': Learning_and_Motivation,
    'Mental Health': Mental_Health
}

df_2019 = pd.DataFrame([category_lists])

In [67]:
df_summary = pd.concat([df_2019, df_2023, df_2024, df_BA_2024, df_2025], ignore_index=True)
df_summary

,STEM Disciplines,Academic Environment,Evaluation,Learning and Motivation,Mental Health
0,"[Science, Physics, Mathematics, Chemistry, Biology, Statistics, Electricity, Computer, Evolution, Dna, Experiment, Cell, Derivative, Space, Graph, Theorem, Trigonometry, Integral, Model, Complex, Chaos, Solid, Number, Hub]","[University, School, Conference, Lab, Studies, Teaching, Student, Degree, Mathematician, Prisoner]","[Analysis, Game, Word, Loop, Market]","[Language, Person]","[Life, Brain, Water, Scientific, Cloud]"
1,"[Algorithm, Biology, Calculation, Demonstration, Equation, Physics, Engineering, Mathematics, Mathematician, Numbers, Performance_in_mathematics, Statistics, Technology, Math_test, Utility_of_mathematics]","[Class, Parent, Teacher, Student, Board, Concept_map, Notes]",[Examination],"[Change_of_plans, Task, Concentration, Confusion, Curiosity, Distraction, Errors, Set_a_goal, I commit, Importance, Intelligence, Store, Motivation, Plan, Problem, Plan, Program, Review, Postpone, Expiration, Challenge, Effort, Solution, Study, Success, Keep_track]","[Anxiety, Failure]"
2,"[Biology, Equation, Physics, Informatics, Mathematics, Statistics, Science, Scientist, Innovation, Neuroscience]","[Teacher, Professor, University]","[Examination, Trial, Test, Average, Assessment]","[Creativity, Curiosity, Fun, Challenge, Model, Know, Development, Passion]","[Anxiety, Attachment, Well being, Cognition, Emotion, Mind, Personality, Psychology, Psychopathology, Psychotherapy, Therapist]"
3,"[Biology, Equation, Physics, Mathematics, Numbers, Statistics, Technology, Graphic, Science, Discovery, Neuroscience]","[Teacher, Student, Professor, School, University]","[Examination, Exercise, Vote]","[Task, Knowledge, Creativity, Curiosity, Challenge, Future, Model, Relation]","[Anxiety, Well being, Cognition, Emotion, Frustration, Mind, Boredom, Personality, Psychology, Therapy, Stress]"
4,"[Analyses, Equation, Physics, Informatics, Mathematics, Numbers, Statistics, Data, Hypothesis, Laboratory, Logic, Research, Science, Scientist, Scientist, Discovery, Experiment, Innovation]","[Companions, Teacher, Parents, School]","[Examination, Vote]","[Knowledge, Creativity, Problem, Programming, Challenge, Future, Rules]","[Anxiety, Art, Well being, Counselling, Failure, Embarrassment, Meditation, Panic, Fear, Stress]"


In [68]:
for col in df_summary.columns:
    if col != 'N':
        df_summary[col] = df_summary[col].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)

df_summary.insert(0, 'N', np.repeat('', len(df_summary)))

In [70]:
latex = df_summary.to_latex(
                           multirow=True, 
                           position='!hbpt',
                           caption='Cue words for different samples.',
                           label='tab: cue_words_summary',
                           column_format ="c|p{2.8cm}|p{3.2cm}|p{2.3cm}|p{3.2cm}|p{2.8cm}",
                           index=False).replace('_', ' ').replace('NaN', '')

latex = latex.replace(r"\begin{table}[!hbpt]", f"\\begin{{table}}[!hbpt]\n\\scriptsize\n\\centering")

with open(f'{rassereno_path}/out/formatting/cue_words_summary_latex.txt', 'w') as f:
    f.write(latex)

## Txt file writing

In [ ]:
with open(f"{rassereno_path}/out/formatting/participants_rassereno.txt", "a") as f:
  f.write("Number of participants for Rassereno article:\n"
  "Experts 2019: 59\n"
  "High school students 2019: 159\n"
  f"High school students 2025: {len(mas_it_liceo_2025)}\n"
  f"Physics students 2025: {len(mas_it_fisica)}\n"
  f"All psychology students: {len(all_psycology_mas_it)} (2025 psy students: {len(mas_it_psychology_2025)})")